# Generate Mixed Split Dataset

This notebook generates patient bundles split between two hospitals using a **mixed strategy**.

The mixed strategy randomly selects between:
- 50% **Temporal** splits
- 25% **Service Type** splits
- 25% **Location** splits

This is recommended for training datasets as it provides diverse splitting patterns.

## Import Libraries and Load Resources

Load the resources, graphs, and helper functions from the main pipeline.

In [1]:
from pathlib import Path
from strategies import SPLIT_STRATEGIES
from dataset_factory import build_split_dataset
from persistence import persist_patient_bundles

## Build Graphs and Load Resources

Load the FHIR resources from compressed NDJSON files and build forward/reverse reference graphs.

In [2]:
from graph_builder import build_fhir_graphs

# Load FHIR resources and build graphs
forward_graph, reverse_graph, resources_map = build_fhir_graphs()

Processing: Condition
Processing: Condition
Processing: Encounter
Processing: Encounter
Processing: Encounter
Processing: Location
Processing: Medication
Processing: MedicationAdministration
Processing: MedicationAdministration
Processing: MedicationDispense
Processing: MedicationDispense
Processing: Medication
Processing: MedicationRequest
Processing: MedicationStatement
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Observation
Processing: Organization
Processing: Patient
Processing: Procedure
Processing: Procedure
Processing: Procedure
Processing: Specimen
Processing: Specimen

Graphs built successfully!
Resource types: ['Condition', 'Encounter', 'Location', 'Medication', 'MedicationAdministration', 'MedicationDispense', 'MedicationRequest', 'MedicationStatement', 'Observation', 'Organization', 'Patient', 'Procedure', 'Specimen'

## Configure Strategy

Select the mixed splitting strategy.

In [3]:
# Select strategy and dataset name
strategy = SPLIT_STRATEGIES["mixed"]
dataset_name = "mixed"
seed = 42  # Set seed for reproducibility

print(f"Strategy: {strategy.__name__}")
print(f"Dataset name: {dataset_name}")
print(f"Random seed: {seed}")
print("\nThis strategy uses:")
print("  - 50% Temporal splits")
print("  - 25% Service Type splits")
print("  - 25% Location splits")

Strategy: mixed_split
Dataset name: mixed
Random seed: 42

This strategy uses:
  - 50% Temporal splits
  - 25% Service Type splits
  - 25% Location splits


## Build Split Dataset

Use the mixed strategy to split patient encounters between hospital_A and hospital_B.

In [4]:
print("Building split dataset...")
patient_bundles_a, patient_bundles_b = build_split_dataset(
    strategy=strategy,
    resources_map=resources_map,
    reverse_graph=reverse_graph,
    forward_graph=forward_graph,
    seed=seed
)

print(f"Created {len(patient_bundles_a)} bundles for hospital_A")
print(f"Created {len(patient_bundles_b)} bundles for hospital_B")
print(f"Patients with split encounters: {len(patient_bundles_b)}")

Building split dataset...
Patient Patient/837b0984-f3bf-5076-baf5-b727decb4bea has only one location; falling back to temporal split.
Created 100 bundles for hospital_A
Created 100 bundles for hospital_B
Patients with split encounters: 100


## Persist Datasets to Disk

Save the split bundles to `input/{dataset_name}/hospital_A` and `input/{dataset_name}/hospital_B`.

In [5]:
print("Persisting bundles...")

persisted_a = persist_patient_bundles(
    patient_bundles_a,
    Path(f"input/{dataset_name}/hospital_A"),
    resources_map
)

persisted_b = persist_patient_bundles(
    patient_bundles_b,
    Path(f"input/{dataset_name}/hospital_B"),
    resources_map
)

print(f"✓ Saved {persisted_a} bundles to input/{dataset_name}/hospital_A")
print(f"✓ Saved {persisted_b} bundles to input/{dataset_name}/hospital_B")
print(f"\nTotal bundles: {persisted_a + persisted_b}")

Persisting bundles...
✓ Saved 100 bundles to input/mixed/hospital_A
✓ Saved 100 bundles to input/mixed/hospital_B

Total bundles: 200
